# Day 17 — The T-test
> Comparing means of two independent groups.

## When to Use a T-test?

- Continuous outcome variable
- Two independent groups
- Approximately normal distribution (or n > 30 by CLT)
- Unknown population variance

## Assumptions
1. Independence of observations
2. Normality (check with Shapiro-Wilk)
3. Homogeneity of variance (check with Levene's test)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')
np.random.seed(42)

# Load or generate data
df = pd.read_csv('../data/session_duration.csv')
before = df[df['group'] == 'before']['session_minutes'].values
after  = df[df['group'] == 'after']['session_minutes'].values

print(f"Before redesign — mean: {before.mean():.3f}, std: {before.std():.3f}, n: {len(before)}")
print(f"After  redesign — mean: {after.mean():.3f}, std: {after.std():.3f}, n: {len(after)}")


In [ ]:
# Step 1: Check normality
stat_b, p_b = stats.shapiro(before[:50])  # Shapiro works best for n<50
stat_a, p_a = stats.shapiro(after[:50])
print(f"Shapiro-Wilk  before: W={stat_b:.4f}, p={p_b:.4f} → {'Normal' if p_b>0.05 else 'Non-normal'}")
print(f"Shapiro-Wilk  after:  W={stat_a:.4f}, p={p_a:.4f} → {'Normal' if p_a>0.05 else 'Non-normal'}")

# Step 2: Check equal variances
lev_stat, lev_p = stats.levene(before, after)
print(f"Levene's test: F={lev_stat:.4f}, p={lev_p:.4f} → {'Equal variances' if lev_p>0.05 else 'Unequal variances'}")


In [ ]:
# Step 3: Run the t-test
equal_var = lev_p > 0.05
t_stat, p_value = stats.ttest_ind(before, after, equal_var=equal_var)
print(f"\nIndependent T-test (equal_var={equal_var})")
print(f"t = {t_stat:.4f}, p = {p_value:.6f}")
print(f"Decision: {'Reject H₀ — redesign changed session duration' if p_value<0.05 else 'Fail to reject H₀'}")

# Cohen's d
n1, n2 = len(before), len(after)
pooled_std = np.sqrt(((n1-1)*before.var(ddof=1) + (n2-1)*after.var(ddof=1)) / (n1+n2-2))
d = (after.mean() - before.mean()) / pooled_std
print(f"Cohen's d = {d:.4f}  ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'} effect)")


In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distributions
for data, label, color in zip([before, after], ['Before', 'After'], ['#4C72B0', '#DD8452']):
    axes[0].hist(data, bins=25, alpha=0.5, color=color, density=True, label=label)
    xd = np.linspace(data.min(), data.max(), 200)
    axes[0].plot(xd, stats.gaussian_kde(data)(xd), color=color, lw=2)
    axes[0].axvline(data.mean(), color=color, linestyle='--', lw=1.5)
axes[0].set_title('Session Duration Distributions')
axes[0].legend()

# Boxplot
axes[1].boxplot([before, after], labels=['Before', 'After'], patch_artist=True,
                boxprops=dict(facecolor='#AED6F1'), medianprops=dict(color='red', lw=2))
axes[1].set_title(f'Boxplot  (p={p_value:.4f})')
axes[1].set_ylabel('Minutes')

plt.suptitle('T-test: Website Redesign Effect on Session Duration', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/03_ttest.png', dpi=150)
plt.show()
